# AI-Powered Prediction of Drug Solubility

**Copyright (c) 2026 Shrikara Kaudambady. All rights reserved.**

This notebook demonstrates a machine learning workflow to predict the aqueous solubility of drug-like molecules. We will represent molecules as numerical 'fingerprints' using the RDKit library and train a Random Forest model to predict their measured solubility (`logS` value).

### 1. Setup and Library Imports

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

### 2. Load the Dataset

We use the well-known Delaney solubility dataset. It contains the SMILES string (a text representation of the molecule) and the experimentally measured `logS` value (log of solubility in mol/L). This dataset is embedded here for portability.

In [ ]:
data_url = 'https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv'
df = pd.read_csv(data_url)

print(f"Dataset loaded with {len(df)} molecules.")
df.head()

### 3. Featurization: From SMILES to Molecular Fingerprints

To use these molecules in a machine learning model, we need to convert them into a numerical format. We will use Morgan Fingerprints, a type of circular fingerprint that represents the presence of specific substructures within a molecule as a vector of 0s and 1s.

In [ ]:
def generate_fingerprint(smiles_string, n_bits=2048):
    """Converts a SMILES string to a Morgan Fingerprint using RDKit."""
    mol = Chem.MolFromSmiles(smiles_string)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    return np.array(fp)

# Apply the function to the SMILES column
df['fingerprint'] = df['smiles'].apply(generate_fingerprint)

# Drop molecules that could not be parsed
df = df.dropna(subset=['fingerprint'])

print("Fingerprints generated successfully.")
print("Example fingerprint for the first molecule:", df['fingerprint'].iloc[0])

### 4. Model Training

We will now train a Random Forest Regressor. The model will learn to map the patterns in the molecular fingerprints to the corresponding solubility (`logS`) value.

In [ ]:
# Prepare data for Scikit-learn
X = np.stack(df['fingerprint'].values)
y = df['measured log solubility in mols per litre'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")

# Initialize and train the model
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
print("\nTraining Random Forest model...")
rf_model.fit(X_train, y_train)
print("Training complete.")

### 5. Model Evaluation

We'll evaluate the model's performance on the held-out test set using R-squared (R²) and a scatter plot of actual vs. predicted values.

In [ ]:
# Make predictions
y_pred = rf_model.predict(X_test)

# Calculate metrics
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"Model Performance on Test Set:")
print(f"  R-squared (R²): {r2:.4f}")
print(f"  Mean Squared Error (MSE): {mse:.4f}")

# Visualize the results
plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], '--', color='red', lw=2)
plt.xlabel("Actual logS")
plt.ylabel("Predicted logS")
plt.title(f"Actual vs. Predicted Solubility (R² = {r2:.4f})")
plt.axis('square')
plt.show()

### 6. Predicting Solubility for New Molecules

Now we can use our trained model to predict the solubility of new molecules that were not in the original dataset. Let's try it on some common drugs.

In [ ]:
new_molecules = {
    'Aspirin': 'CC(=O)OC1=CC=CC=C1C(=O)O',
    'Ibuprofen': 'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Paclitaxel (a complex, likely insoluble drug)': 'CC1=C(C(=O)C2(C(C1)OC3(C(C2=O)C(C4(C(C3O)CC(C(C4(C)C)O)C(=O)C5=CC=CC=C5)OC(=O)C6=CC=CC=C6)OC(=O)C)O)O)OC(=O)C(C(C7=CC=CC=C7)NC(=O)C8=CC=CC=C8)O'
}

print("--- Predictions for New Molecules ---")
for name, smiles in new_molecules.items():
    fp = generate_fingerprint(smiles)
    if fp is not None:
        prediction = rf_model.predict(fp.reshape(1, -1))
        print(f"  -> Predicted logS for {name}: {prediction[0]:.4f}")
    else:
        print(f"Could not parse SMILES for {name}")